# Donut distillation results

Compare the strict macro field F1 and parameter counts saved by `evaluate_distillation.py`.

In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

RESULT_PATH = Path("results/distillation_evaluation.json")

In [ ]:
record = json.loads(RESULT_PATH.read_text())

comparison = pd.DataFrame(
    {
        name: {
            "strict_macro_field_f1": record[name]["score"],
            "total_parameters": record[name]["parameters"]["total"],
            "encoder_parameters": record[name]["parameters"]["encoder"],
            "decoder_parameters": record[name]["parameters"]["decoder"],
        }
        for name in ("teacher", "student")
    }
).T
comparison

In [ ]:
teacher_parameters = comparison.loc["teacher", "total_parameters"]
student_parameters = comparison.loc["student", "total_parameters"]

pd.Series(
    {
        "parameters_removed": teacher_parameters - student_parameters,
        "parameter_reduction": 1 - student_parameters / teacher_parameters,
        "score_change": (
            comparison.loc["student", "strict_macro_field_f1"]
            - comparison.loc["teacher", "strict_macro_field_f1"]
        ),
    }
)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4))

comparison["strict_macro_field_f1"].plot.bar(ax=axes[0], rot=0)
axes[0].set(title="Extraction quality", ylabel="Strict macro field F1", ylim=(0, 1))

parameter_columns = ["encoder_parameters", "decoder_parameters"]
(comparison[parameter_columns] / 1e6).plot.bar(ax=axes[1], rot=0)
axes[1].set(title="Model parameters", ylabel="Millions of parameters")

plt.tight_layout()